In [ ]:
// load the formatting setup, the visualization functions, and the style sheet
#load "../setup.fsx"
#load "../visuals.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

# Map Coloring

This short notebook shows how *map coloring* can be formulated as a *constraint satisfaction problem*.
In map coloring, the goal is to color the states shown in a given map such that no two bordering states 
share the same color.  As an example, consider the map of Australia that is shown below.  Australia
has seven different states:
* [Western Australia](https://en.wikipedia.org/wiki/Western_Australia),
* [Northern Territory](https://en.wikipedia.org/wiki/Northern_Territory),
* [South Australia](https://en.wikipedia.org/wiki/South_Australia),
* [Queensland](https://en.wikipedia.org/wiki/Queensland),
* [New South Wales](https://en.wikipedia.org/wiki/New_South_Wales),
* [Victoria](https://en.wikipedia.org/wiki/Victoria_(Australia)), and
* [Tasmania](https://en.wikipedia.org/wiki/Tasmania).

As Tasmania is an island that does not share a border with any of the other states, it can be colored with any given color.

<img src="australia.png" alt="map of australia" width="500">

We import the backtracking solver with animation.  This also imports the notebook `02-Backtracking-Constraint-Solver.ipynb`, which defines the types `Constraint<'V>` and `CSP<'V>` and the function `different` that creates a constraint of the form $x \not= y$.

In [ ]:
importNotebook "Backtrack-Solver-Animate.ipynb"

The function `mapColoringCSP` takes no arguments.  It returns a CSP that encodes the *map coloring problem* for Australia.  The values are the names of the colors.

In [ ]:
let mapColoringCSP () : CSP<string> =
    { Variables   = ["WA"; "NSW"; "T"; "V"; "NT"; "SA"; "Q"]
      Values      = ["red"; "blue"; "green"]
      Constraints = [different "WA" "NT";  different "WA" "SA"
                     different "NT" "SA";  different "NT" "Q"
                     different "SA" "Q";   different "SA" "NSW"; different "SA" "V"
                     different "Q"  "NSW"; different "NSW" "V"] }

In [ ]:
let csp = mapColoringCSP ()
csp

As there are $3$ different values and $7$ different variables, there are $3^7 = 2187$ different ways to assign values to the variables.

In [ ]:
pown 3 7

## Visualization

The function `showSolution` takes two arguments:
 * `Solution` is a partial variable assignment, i.e. it is a map
   assigning colors to some of the variables,
 * `width` specifies the width of the image that will be created.

It returns a picture of the map of Australia where the states are colored according to `Solution`.  The function `australiaMap` is defined in the file `visuals.fsx`, which has been loaded in the first cell.

In [ ]:
let showSolution (Solution: Map<string, string>) (width: string) =
    australiaMap Solution width

In [ ]:
showSolution (Map [("WA", "red"); ("NT", "green")]) "40%"

We solve the problem with the animated solver.  The function `display` is provided by the notebook kernel.  It displays its argument in the output of the current cell.  The expression `display x |> ignore` discards the value returned by `display`.

In [ ]:
let Solution1 = solveAnimated (fun A -> display (showSolution A "20%") |> ignore) csp
Solution1

Below, we have changed the ordering of the variables.  With this new ordering, the constraint solver never needs to backtrack. 

In [ ]:
let mapColoringCSP2 () : CSP<string> =
    { mapColoringCSP () with Variables = ["WA"; "NT"; "SA"; "Q"; "NSW"; "V"; "T"] }

The expression `{ r with Field = value }` creates a copy of the record `r` where the given field has been replaced.

In [ ]:
let Solution2 = solveAnimated (fun A -> display (showSolution A "20%") |> ignore) (mapColoringCSP2 ())
Solution2